In [1]:
import math
import numpy as np

In [2]:
activation_func = 1 / (1 + math.exp(1.5))
# print(activation_func)

w1 = -1
w2 = -1
b = 1.5
if b>= 0:
    print(b)
if w2+b >= 0:
    print(w2, b)
if w1+b >= 0:
    print(w1, b)
if w1+w2+b < 0:
    print(w1, w2, b)

1.5
-1 1.5
-1 1.5
-1 -1 1.5


In [3]:
# # Designing a 2-layer perceptron network to solve the 3-input **even-parity** problem.
# # Even parity: output = 1 when the number of 1s in (x1,x2,x3) is even (0 or 2), else 0.
# # Strategy:
# # - Hidden layer: 4 perceptrons, each detects one input pattern that should map to output 1:
# #   patterns: 000, 011, 101, 110
# # - Final output perceptron: OR of hidden neurons (outputs 1 if any hidden neuron fires).
# #
# # Construction rule for a hidden neuron that detects a specific binary pattern p (length 3):
# # - weight_i = +1 if p_i == 1, else -1
# # - bias = - (number_of_ones_in_p) + 0.5
# # This makes the net input = 0.5 for the matching pattern and (we'll verify) < 0 for other inputs.
# #
# # Then final neuron: weights = [1,1,1,1], bias = -0.5 (so any single hidden 1 yields final output 1).
# # We'll test all 8 input combinations and print the outputs of hidden neurons and final output.
#
# import itertools
# import pandas as pd
#
# # Define target patterns for even parity (0 or 2 ones)
# patterns = [(0,0,0),(0,1,1),(1,0,1),(1,1,0)]
#
# def make_detector(pattern):
#     weights = [1 if bit==1 else -1 for bit in pattern]
#     bias = -sum(pattern) + 0.5
#     return weights, bias
#
# hidden = [make_detector(p) for p in patterns]
#
# def perceptron_output(weights, bias, inputs):
#     net = sum(w*x for w,x in zip(weights, inputs)) + bias
#     return 1 if net >= 0 else 0, net
#
# rows = []
# for x in itertools.product([0,1], repeat=3):
#     hidden_outputs = []
#     hidden_nets = []
#     for (w,b) in hidden:
#         out, net = perceptron_output(w,b,x)
#         hidden_outputs.append(out)
#         hidden_nets.append(net)
#     # final OR perceptron
#     final_weights = [1,1,1,1]
#     final_bias = -0.5
#     final_net = sum(h*fw for h,fw in zip(hidden_outputs, final_weights)) + final_bias
#     final_out = 1 if final_net >= 0 else 0
#     # true even parity label
#     true_label = 1 if (sum(x) % 2 == 0) else 0
#     rows.append({
#         "x1": x[0], "x2": x[1], "x3": x[2],
#         "hidden_outputs": hidden_outputs,
#         "hidden_nets": [round(n,3) for n in hidden_nets],
#         "final_net": round(final_net,3),
#         "final_out": final_out,
#         "true_label": true_label
#     })
#
# df = pd.DataFrame(rows)
# tools.display_dataframe_to_user("3-bit Parity Perceptron Test", df)
# df


In [5]:
# Perceptron vs. MLP on 3-bit parity problem
# Create dataset: all 3-bit inputs
X = np.array([[int(x) for x in f"{i:03b}"] for i in range(8)])  # shape (8,3)
print(X)

# Parity: 1 if number of 1s is even, else 0
y = np.array([1 if x.sum()%2==0 else 0 for x in X]).reshape(-1,1)

print("Inputs (X) and targets (y):")
for xi, yi in zip(X, y):
    print(xi, yi[0])

# --- Single-layer perceptron (hard threshold) ---
def train_perceptron(X, y, epochs=100, lr=1.0):
    # Add bias term
    Xb = np.hstack([np.ones((X.shape[0],1)), X])
    w = np.zeros((Xb.shape[1],1))
    for epoch in range(epochs):
        errors = 0
        for xi, target in zip(Xb, y):
            net = float(xi @ w)
            out = 1 if net >= 0 else 0
            e = target - out
            if e != 0:
                w += lr * e * xi.reshape(-1,1)
                errors += 1
        if errors == 0:
            return w, True, epoch+1  # converged
    return w, False, epochs

w, converged, epochs = train_perceptron(X, y, epochs=1000, lr=1.0)
# print("\nSingle-layer perceptron results:")
# print("Converged:", converged, "after epochs:", epochs)
# print("Weights (bias first):", w.ravel())

# Evaluate perceptron
Xb = np.hstack([np.ones((X.shape[0],1)), X])
preds = (Xb @ w >= 0).astype(int)
print("Perceptron predictions vs targets:")
for xi, p, t in zip(X, preds, y):
    print(xi, int(p[0]), int(t[0]))

# --- Small MLP (one hidden layer) ---
# Architecture: 3 inputs -> 4 hidden (sigmoid) -> 1 output (sigmoid)
def sigmoid(z): return 1/(1+np.exp(-z))
def sigmoid_deriv(a): return a*(1-a)

rng = np.random.default_rng(1)
n_in = 3; n_h = 4; n_out = 1

W1 = rng.normal(scale=0.5, size=(n_in, n_h))
b1 = np.zeros((1, n_h))
W2 = rng.normal(scale=0.5, size=(n_h, n_out))
b2 = np.zeros((1, n_out))

def forward(X):
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)
    return z1, a1, z2, a2

# Train with simple batch gradient descent
lr = 1.0
for epoch in range(10000):
    z1, a1, z2, a2 = forward(X)
    loss = ((a2 - y)**2).mean()
    # Backprop (MSE)
    dA2 = 2*(a2 - y)/y.size
    dZ2 = dA2 * sigmoid_deriv(a2)
    dW2 = a1.T @ dZ2
    db2 = dZ2.sum(axis=0, keepdims=True)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * sigmoid_deriv(a1)
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0, keepdims=True)
    # Update
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1
    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, loss {loss:.6f}")
    if loss < 1e-6:
        print(f"Converged at epoch {epoch}, loss {loss:.8f}")
        break

# Evaluate MLP
_, _, _, a2 = forward(X)
preds_mlp = (a2 >= 0.5).astype(int)
print("\nMLP predictions vs targets:")
for xi, p, t, prob in zip(X, preds_mlp, y, a2):
    print(xi, int(p[0]), int(t[0]), f"(prob {prob[0]:.4f})")

acc = (preds_mlp == y).mean()
# print(f"\nMLP accuracy: {acc*100:.1f}%")

# Show final weights (rounded)
# print("\nFinal MLP weights (rounded):")
# print("W1 (3x4):\n", np.round(W1,3))
# print("b1 (1x4):\n", np.round(b1,3))
# print("W2 (4x1):\n", np.round(W2,3))
# print("b2 (1x1):\n", np.round(b2,3))


[[0 0 0]
 [0 0 1]
 [0 1 0]
 [0 1 1]
 [1 0 0]
 [1 0 1]
 [1 1 0]
 [1 1 1]]
Inputs (X) and targets (y):
[0 0 0] 1
[0 0 1] 0
[0 1 0] 0
[0 1 1] 1
[1 0 0] 0
[1 0 1] 1
[1 1 0] 1
[1 1 1] 0


TypeError: only 0-dimensional arrays can be converted to Python scalars